# Notebook 4: Clickbait Detector Demo
Type a headline, press **Predict**, and get the predicted class plus the model's clickbait confidence.

Note for testers: Run the cells top to bottom. 

## 1. Setup

In [ ]:
!pip install -q transformers torch ipywidgets scikit-learn joblib
import os, joblib
import numpy as np

## 2. Model paths 


In [ ]:
DISTILBERT_DIR = "./distilbert_clickbait"   # folder saved with model.save_pretrained() + tokenizer.save_pretrained()
TFIDF_LR_PATH  = "./tfidf_lr.joblib"        # sklearn Pipeline, OR a dict {"vectorizer": ..., "model": ...}
MAX_LEN = 64

## 3. Load whichever models are available

In [ ]:
models = {}

# --- DistilBERT ---
if os.path.isdir(DISTILBERT_DIR):
    import torch
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    device = "cuda" if torch.cuda.is_available() else "cpu"
    bert_tok = AutoTokenizer.from_pretrained(DISTILBERT_DIR)
    bert_model = AutoModelForSequenceClassification.from_pretrained(DISTILBERT_DIR).to(device).eval()
    models["DistilBERT"] = "bert"
else:
    print(f"DistilBERT folder not found: {DISTILBERT_DIR}")

# --- TF-IDF + Logistic Regression ---
if os.path.exists(TFIDF_LR_PATH):
    tfidf_obj = joblib.load(TFIDF_LR_PATH)
    models["TF-IDF + Logistic Regression"] = "tfidf"
else:
    print(f"TF-IDF model not found: {TFIDF_LR_PATH}")

print("Available models:", list(models))

DistilBERT folder not found: ./distilbert_clickbait
TF-IDF model not found: ./tfidf_lr.joblib
Available models: []


## 4. Prediction functions
Label convention (same as the cleaned data): **0 = non-clickbait, 1 = clickbait**.

In [ ]:
def clickbait_prob(headline, model_name):
    """Return P(clickbait) as a float in [0, 1]."""
    kind = models[model_name]
    if kind == "bert":
        enc = bert_tok(headline, return_tensors="pt", truncation=True, max_length=MAX_LEN).to(device)
        with torch.no_grad():
            logits = bert_model(**enc).logits
        return torch.softmax(logits, dim=-1)[0, 1].item()
    else:
        if isinstance(tfidf_obj, dict):
            X = tfidf_obj["vectorizer"].transform([headline])
            return float(tfidf_obj["model"].predict_proba(X)[0, 1])
        return float(tfidf_obj.predict_proba([headline])[0, 1])

def predict(headline, model_name):
    p = clickbait_prob(headline, model_name)
    return ("Clickbait" if p >= 0.5 else "Non-clickbait"), p

## 5. Demo UI

In [ ]:
import ipywidgets as w
from IPython.display import display, HTML, clear_output

DISCLAIMER = ("The percentage is a <b>model confidence-based estimate</b>, "
              "not a verified measure of how misleading the headline is.")

headline_box = w.Textarea(placeholder="Type or paste a news headline here...",
                          layout=w.Layout(width="650px", height="70px"))
model_pick = w.Dropdown(options=list(models) or ["(no model loaded)"],
                        description="Model:", layout=w.Layout(width="400px"))
predict_btn = w.Button(description="Predict", button_style="primary", icon="search")
out = w.Output()

def on_predict(_):
    with out:
        clear_output()
        text = headline_box.value.strip()
        if not models:
            print("No model loaded. Check the paths in cell 2."); return
        if not text:
            print("Please enter a headline first."); return
        label, p = predict(text, model_pick.value)
        color = "#b3261e" if label == "Clickbait" else "#1b6e3c"
        display(HTML(f"""
        <div style='font-family:sans-serif;padding:12px 16px;border-left:5px solid {color};background:#f6f6f6;max-width:650px'>
          <div style='color:#555'>Headline</div>
          <div style='margin-bottom:8px'><i>{text}</i></div>
          <div><b>Predicted class:</b> <span style='color:{color};font-weight:bold'>{label}</span></div>
          <div><b>Clickbait confidence:</b> {p*100:.1f}%</div>
          <div style='color:#555;font-size:0.85em;margin-top:8px'>{DISCLAIMER}</div>
        </div>"""))

predict_btn.on_click(on_predict)
display(w.VBox([headline_box, w.HBox([model_pick, predict_btn]), out]))

### Optional: quick sanity-check headlines some test that can be used to see if clickbait or not

Paste any of these into the box above.
- `You Won't Believe What This Dog Did Next`
- `Central bank raises interest rates by 0.25 percentage points`
- `10 Reasons Your Morning Routine Is Ruining Your Life`